In [ ]:
!pip install -q transformers pillow accelerate

import torch
from PIL import Image
import requests
from io import BytesIO
from typing import List
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID = "nlpconnect/vit-gpt2-image-captioning"
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device)
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


gen_kwargs = {
    "max_length": 20,
    "num_beams": 3,
    "no_repeat_ngram_size": 2
}

def load_image(img_path_or_url: str) -> Image.Image:
    if img_path_or_url.lower().startswith(("http://", "https://")):
        resp = requests.get(img_path_or_url, timeout=30)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        return img
    else:
        return Image.open(img_path_or_url).convert("RGB")

@torch.no_grad()
def caption_image(img_path_or_url: str) -> str:
    image = load_image(img_path_or_url)
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    output_ids = model.generate(pixel_values, **gen_kwargs)
    caption = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return caption

@torch.no_grad()
def caption_batch(images: List[str], batch_size: int = 8) -> List[str]:
    captions = []
    for i in range(0, len(images), batch_size):
        batch_paths = images[i:i+batch_size]
        batch_imgs = [load_image(p) for p in batch_paths]
        pixel_values = processor(images=batch_imgs, return_tensors="pt", padding=True).pixel_values.to(device)
        out_ids = model.generate(pixel_values, **gen_kwargs)
        caps = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        captions.extend([c.strip() for c in caps])
    return captions


sample_url = "https://images.unsplash.com/photo-1519681393784-d120267933ba?w=1200"
print("Caption:", caption_image(sample_url))

batch_imgs = [
    "https://images.unsplash.com/photo-1500530855697-b586d89ba3ee?w=1200",
    "https://images.unsplash.com/photo-1507525428034-b723cf961d3e?w=1200",
]
for cap in caption_batch(batch_imgs):
    print("•", cap)